# AI Translation Matcher – Visual Similarity Only
This notebook evaluates the search for similar images based solely on visual content, using MobileNetV2 and `NearestNeighbors`. It relies on the vectors already extracted in `output/image_vectors.pkl`.

It evaluates the quality and speed of pure visual search.

In [ ]:
import sys
import os
sys.path.append(os.path.abspath("../../src"))

import pickle
import numpy as np
import time
from sklearn.neighbors import NearestNeighbors
from fetch import download_file_from_space
from image_features_mobilenet import extract_visual_features
from visual_matcher import build_index, search_similar_images
from PIL import Image
from IPython.display import display
import io
import ipywidgets as widgets
from IPython.display import display as ipy_display


In [ ]:
test_dir = "../../AI_matcher/tests"
valid_extensions = (".jpg", ".jpeg", ".png")

available_files = sorted([
    f for f in os.listdir(test_dir)
    if os.path.isfile(os.path.join(test_dir, f))
    and f.lower().endswith(valid_extensions)
    and not f.startswith(".DS_Store")
])

dropdown = widgets.Dropdown(
    options=available_files,
    description="File:",
    disabled=False
)

ipy_display(dropdown)

In [ ]:
query_path = os.path.join(test_dir, dropdown.value)
vector_path = "../../output/image_vectors_mobilenet.pkl"
top_k = 5

with open(query_path, "rb") as f:
    query_vec = extract_visual_features(f.read())

In [ ]:
with open(vector_path, "rb") as f:
    data = pickle.load(f)

keys = data["keys"]
vectors = data["vectors"]

index = NearestNeighbors(n_neighbors=top_k, metric='euclidean')
index.fit(vectors)

In [ ]:
start = time.time()
distances, indices = index.kneighbors([query_vec])
end = time.time()

results = [(keys[i], round(100 - distances[0][j], 2)) for j, i in enumerate(indices[0])]

print("Top image-based matches:")
for file, score in results:
    print(f"{file} — {score}%")

print(f"Execution time: {end - start:.2f} seconds")

In [ ]:
from IPython.display import Image as IPyImage
import fitz
from docx import Document

def display_document_preview(file_key: str):
    print(f"\nPreview: {file_key}")
    try:
        content = download_file_from_space(file_key)

        if file_key.lower().endswith((".jpg", ".jpeg", ".png")):
            img = Image.open(io.BytesIO(content))
            display(img)

        elif file_key.lower().endswith(".pdf"):
            doc = fitz.open(stream=content, filetype="pdf")
            pix = doc[0].get_pixmap(dpi=150)
            ipy_img = IPyImage(data=pix.tobytes("png"))
            display(ipy_img)

        elif file_key.lower().endswith(".docx"):
            doc = Document(io.BytesIO(content))
            for rel in doc.part._rels:
                rel_obj = doc.part._rels[rel]
                if "image" in rel_obj.target_ref:
                    img_data = rel_obj.target_part.blob
                    img = Image.open(io.BytesIO(img_data))
                    display(img)
                    break
            else:
                print("[INFO] No image found in the .docx file.")

        else:
            print("[SKIP] Unsupported file type for preview.")
    except Exception as e:
        print(f"[ERROR] Could not display {file_key}: {e}")

In [ ]:
print("\nDisplaying matched document previews:")
for file_key, _ in results:
    display_document_preview(file_key)

print(f"\nExecution time: {end - start:.2f} seconds")